# Bài 33 — chuỗi luyện Easy · LeetCode #169 — Majority Element

**Giáo trình Jupyter Notebook tiếng Việt cho người mới bắt đầu · Python 3**  
**Link đề:** https://leetcode.com/problems/majority-element/ · **Mức độ:** Easy · **Chủ điểm:** Array · Counting · Boyer–Moore Majority Vote

**Kết quả bạn đạt được sau bài học**
1. Phát biểu được chính xác input, output, và những điều kiện đề bảo đảm; phân biệt dữ liệu thật với ví dụ mô phỏng.
2. Tự diễn giải công cụ Python liên quan bằng chương trình mini; không ghi nhớ cú pháp một cách máy móc.
3. Tự nghĩ ra một cách giải dễ hiểu, thấy rõ điểm hạn chế, rồi suy ra cách chính tốt hơn.
4. Lập bảng và chạy chương trình `trace_*` để thấy trạng thái thay đổi từng bước, kể cả nhánh đặc biệt.
5. Giải thích được tính đúng bằng **bất biến (invariant)**, tự suy luận Time/Space Complexity.
6. Viết lại `class Solution` để nộp LeetCode và tự giải 6 bài tập, có gợi ý ba mức và đáp án cuối.

> **Cách học:** Đọc một markdown → tự dự đoán kết quả code → chạy cell → so sánh → tự giải thích vì sao. Các cell TODO chỉ định nghĩa hàm, không được chạy tự động.


## 1. Đọc đề và hiểu “quá bán” chính xác

Cho mảng số nguyên `nums` có độ dài `n`, tìm **majority element**: giá trị xuất hiện **nhiều hơn** `floor(n/2)` lần (dấu `>` nghiêm ngặt, không phải `>=`). Đề **bảo đảm luôn tồn tại** một majority element; `1 <= n <= 5*10^4`, giá trị phần tử có thể âm hoặc dương. Ví dụ `[3,2,3] → 3`; `[2,2,1,1,1,2,2] → 2`.

Giả sử n=7 thì một giá trị phải có ít nhất 4 lần mới “quá bán”. Nếu n=6 thì xuất hiện đúng 3 lần **không đủ**. Hãy phân biệt “giá trị xuất hiện nhiều nhất” (mode) với “giá trị quá bán”: giá trị nhiều nhất có thể chỉ xuất hiện 2/5, nhưng majority nếu tồn tại luôn là duy nhất.

**Follow-up:** Có cách **Time O(n), Extra Space O(1)** không? Ta sẽ đi từ đếm tần suất bằng dict/Counter (dễ hiểu) tới thuật toán bỏ từng cặp khác nhau: **Boyer–Moore Majority Vote**.


## 2. Thiết lập môi trường để chạy notebook

Notebook dùng **Python 3** chuẩn, không cần cài thêm package. Hãy chạy **từ trên xuống dưới** để các tên hàm đã được định nghĩa trước khi gọi. Không copy `print(...)` vào LeetCode; các dòng in ở đây chỉ phục vụ việc học.

**Phân biệt ba lớp:** (1) demo nhỏ để hiểu công cụ, (2) hàm minh họa giải bài trong notebook, (3) `class Solution` đúng chữ ký để nộp trên trang đề. Với bài #169, ta sẽ học kỹ thuật **Array · Counting · Boyer–Moore Majority Vote** theo đúng thứ tự này.

Từng code cell sau đây cần cho ra output ngắn và có thể kiểm tra bằng mắt. Trong cell kiểm thử, `assert` không có output nghĩa là điều kiện đã đúng; nếu sai, Python báo `AssertionError`.


In [1]:
from __future__ import annotations
from collections import Counter
import random
print('Mảng:',[2,2,1,1,1,2,2])
print('Counter:',Counter([2,2,1,1,1,2,2]))


Mảng: [2, 2, 1, 1, 1, 2, 2]
Counter: Counter({2: 4, 1: 3})


### 2.1. `list`, `len`, `for` và số lượng phần tử

Độ dài n lấy từ `len(nums)`. Mảng có thể chứa số âm, số 0, các số trùng. Vòng `for x in nums` lấy từng giá trị; chỉ dùng `enumerate(nums)` nếu cần cả chỉ số. Không nhầm **chỉ số i** với **giá trị x** khi tính số lần xuất hiện.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [2]:
nums=[2,2,1,1,1,2,2]
print('n=',len(nums),'ngưỡng=',len(nums)//2)
for i,x in enumerate(nums[:3]):print('vị trí',i,'giá trị',x)


n= 7 ngưỡng= 3
vị trí 0 giá trị 2
vị trí 1 giá trị 2
vị trí 2 giá trị 1


### 2.2. `//` và điều kiện “lớn hơn một nửa”

`n//2` là chia nguyên lấy phần nguyên, ví dụ 7//2=3. Majority có số lần `>3`, tức từ 4 trở lên. Với n=6, `n//2=3`, cần từ 4. Không nhầm `>=` với `>`; hai giá trị cùng 3/6 không có majority.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [3]:
for n in [1,2,3,6,7]:
    print('n=',n,'mốc>',n//2,'=> cần',n//2+1,'lần')


n= 1 mốc> 0 => cần 1 lần
n= 2 mốc> 1 => cần 2 lần
n= 3 mốc> 1 => cần 2 lần
n= 6 mốc> 3 => cần 4 lần
n= 7 mốc> 3 => cần 4 lần


### 2.3. `dict` đếm tần suất từng giá trị

`freq={}` là bảng key→count. Với mỗi x: `freq[x] = freq.get(x,0)+1`. `.get(x,0)` trả 0 nếu key chưa có, tránh KeyError. Cách này rất trực giác nhưng số lượng key có thể tăng đến n, nên Space O(n).

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [4]:
freq={}
for x in [2,1,2,3,2]:
    freq[x]=freq.get(x,0)+1
    print('đọc',x,'->',freq)
assert freq=={2:3,1:1,3:1}


đọc 2 -> {2: 1}
đọc 1 -> {2: 1, 1: 1}
đọc 2 -> {2: 2, 1: 1}
đọc 3 -> {2: 2, 1: 1, 3: 1}
đọc 2 -> {2: 3, 1: 1, 3: 1}


### 2.4. `collections.Counter` là dict đếm chuyên dụng

`Counter(nums)` cho bảng tần suất rất gọn; `Counter(nums).most_common(1)` trả cặp (giá trị, số lần) xuất hiện nhiều nhất. Nhưng để hiểu thuật toán, ta vẫn viết dict thủ công trước. Giải pháp này O(n) time trung bình và O(k) bộ nhớ với k giá trị phân biệt.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [5]:
nums=[3,2,3,3,1]
c=Counter(nums)
print(c, 'phổ biến nhất:',c.most_common(1))
assert c[3]==3


Counter({3: 3, 2: 1, 1: 1}) phổ biến nhất: [(3, 3)]


### 2.5. Hai phép kiểm so sánh `==` và `!=`

Boyer–Moore không cần biết hai số cách nhau bao nhiêu: chỉ cần so `x == candidate`. Giống thì tăng phiếu; khác thì giảm. Dùng `==` (so sánh), không dùng `=` (phép gán) trong điều kiện `if`.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [6]:
candidate=2
for x in [2,3,2,-2]:
    print(x,'== candidate?',x==candidate)


2 == candidate? True
3 == candidate? False
2 == candidate? True
-2 == candidate? False


### 2.6. Bộ đếm `count` có nghĩa gì?

Trong kỹ thuật này `count` **không phải tổng số lần candidate xuất hiện trong mảng**. Nó là số phiếu chưa bị triệt tiêu sau khi ghép các cặp có giá trị khác nhau trong phần đã đọc. Vì vậy có thể từ 3 giảm xuống 2 dù không xóa phần tử nào.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [7]:
candidate=2;count=0
for x in [2,2,1,2,3]:
    count+=1 if x==candidate else -1
    print('x=',x,'phiếu còn=',count)


x= 2 phiếu còn= 1
x= 2 phiếu còn= 2
x= 1 phiếu còn= 1
x= 2 phiếu còn= 2
x= 3 phiếu còn= 1


### 2.7. Bất biến và phép xóa một cặp khác nhau

Nếu một giá trị xuất hiện nhiều hơn n/2, việc bỏ đồng thời hai phần tử **khác nhau** không thể khiến majority ấy mất tính hơn một nửa trong phần còn lại: nếu một phần tử là majority thì nó chỉ mất một phiếu, còn tổng giảm hai. Ta sẽ biến ý tưởng “triệt tiêu cặp” thành biến `candidate, count`.

**Tự dự đoán trước khi chạy:** Giá trị nào sẽ thay đổi, giá trị nào giữ nguyên? Khi chạy xong, thử thay một đầu vào nhỏ và giải thích kết quả theo từng phép tính.


In [8]:
seq=[2,2,1,1,1,2,2]
seq2=seq.copy();seq2.remove(2);seq2.remove(1)
print('ban đầu:',Counter(seq))
print('bỏ cặp 2 và 1:',Counter(seq2))
assert seq2.count(2)>len(seq2)//2


ban đầu: Counter({2: 4, 1: 3})
bỏ cặp 2 và 1: Counter({2: 3, 1: 2})


## 3. Nền tảng tư duy — từ ví dụ nhỏ đến bài toán

Bảy thí nghiệm ở trên không phải những mẹo rời rạc: chúng chính là các khối xây dựng cho bài #169. Hãy kiểm tra lần lượt: **dữ liệu đang nằm ở đâu, biến nào lưu thông tin đã biết, bước nào khiến ta tiến tới đáp án?**

Khi học thuật toán, đừng chỉ hỏi “code này chạy gì?”; hãy hỏi **tại sao không cần nhớ phần thông tin đã bỏ đi** và vì sao ta không bỏ sót lời giải. Phần sau sẽ chạy một số ví dụ rất nhỏ trước khi ghép thành hàm hoàn chỉnh.


In [9]:
def frequencies_169(nums):
    table={}
    for x in nums:table[x]=table.get(x,0)+1
    return table
for arr in ([3,2,3],[2,2,1,1,1,2,2],[-1,-1,2]):
    print(arr,'=>',frequencies_169(arr))


[3, 2, 3] => {3: 2, 2: 1}
[2, 2, 1, 1, 1, 2, 2] => {2: 4, 1: 3}
[-1, -1, 2] => {-1: 2, 2: 1}


### Câu hỏi gợi mở trước khi tối ưu

- Nếu có một số xuất hiện >n/2 lần, có thể có một số thứ hai cũng >n/2 không? Vì sao?
- Ta thực sự cần giữ toàn bộ bảng tần suất không, nếu đã biết lời hứa “majority luôn tồn tại”?
- Nếu xung đột hai giá trị khác nhau, có thể xóa một cặp mà không đổi majority cuối không?

Chưa cần thuộc tên kỹ thuật **Array · Counting · Boyer–Moore Majority Vote**. Trước tiên hãy tưởng tượng bạn chỉ có bút và giấy: bạn đọc input theo thứ tự nào, giữ lại những gì, điều gì buộc bạn dừng? Câu trả lời sẽ trở thành các biến và điều kiện `if`/`while`.


### Quy ước chạy cell và cách tự kiểm tra

- Đọc thứ tự từ trên xuống: các hàm helper trong phần này chỉ dùng để dựng dữ liệu hoặc hiển thị; đáp án submit sẽ không phụ thuộc chúng.
- `print` là quan sát, `return` là kết quả hàm. Dùng `assert actual == expected` để kiểm chứng, không thay thế một phép kiểm bằng việc nhìn thấy chữ PASS.
- Khi thấy nhiều thuật toán, chúng có **tên hàm khác nhau** (`baseline_*`, `solve_*`, `trace_*`) để tránh ghi đè logic. Không thay đổi input trừ khi đề cho phép.
- Đừng chạy hàm TODO trước khi tự hoàn thiện: hàm được để `pass`/`NotImplementedError` có chủ đích.

**Nguyên tắc quan trọng:** ví dụ và test là bằng chứng thử nghiệm; lập luận về bất biến mới giúp giải thích vì sao thuật toán đúng cho mọi input hợp lệ.


### 3.1. Tính tần suất và kiểm ngưỡng bằng tay

Với `[2,2,1,1,1,2,2]`, n=7, `n//2=3`. Ta đếm được `2→4`, `1→3`, chỉ 2 thỏa `>3`. Counter là cách kiểm tra chính xác, nhưng vẫn lưu ít nhất một key cho mỗi giá trị phân biệt.

Chạy đoạn code dưới rồi thử tự diễn đạt **trạng thái trước – thao tác – trạng thái sau**. Nếu chỉ nhớ kết quả in ra mà không giải thích được sự chuyển đổi thì nên đọc lại ví dụ.


In [10]:
arr=[2,2,1,1,1,2,2]
threshold=len(arr)//2
for x, count in frequencies_169(arr).items():
    print('số',x,'xuất hiện',count,'lần, quá bán?',count>threshold)


số 2 xuất hiện 4 lần, quá bán? True
số 1 xuất hiện 3 lần, quá bán? False


### 3.2. Triệt tiêu một cặp khác nhau, theo dõi phiếu

Hãy đặt một số nổi trội và một số đối lập lên bàn, gạch đi từng cặp khác nhau. Nếu số nổi trội thật sự quá bán, số ấy không thể bị gạch hết. Thuật toán Boyer–Moore thực hiện ghép cặp **ngầm** chứ không cần xây danh sách các cặp bị xóa.

Đây là bước nối giữa ví dụ nhỏ và code thuật toán: những biến xuất hiện trong cell tiếp theo sẽ tái xuất hiện trong `solve_*` hoặc `trace_*` ở dưới.


In [11]:
nums=[2,2,1,1,1,2,2]
candidate=None;count=0
for x in nums:
    if count==0:candidate=x
    count+=1 if x==candidate else -1
    print('x=',x,'-> candidate=',candidate,'count=',count)
print('ứng viên cuối=',candidate)


x= 2 -> candidate= 2 count= 1
x= 2 -> candidate= 2 count= 2
x= 1 -> candidate= 2 count= 1
x= 1 -> candidate= 2 count= 0
x= 1 -> candidate= 1 count= 1
x= 2 -> candidate= 1 count= 0
x= 2 -> candidate= 2 count= 1
ứng viên cuối= 2


## 4. Cách giải trực giác / Baseline

Cách đầu tiên: đếm bằng dict/Counter rồi tìm phần tử có tần suất > n//2. Vì đề bảo đảm có majority, chắc chắn tìm ra. Ta cũng dùng nó làm **oracle** để so với thuật toán tối ưu; việc chạy oracle phải độc lập, không sử dụng logic bỏ phiếu để tránh chia sẻ cùng một bug.

Ta cố ý viết phiên bản **dễ nghĩ nhất trước**, dù đôi khi nó dùng thêm bộ nhớ hoặc thao tác dư thừa. Lợi ích của baseline: nhìn được định nghĩa bài toán thành code, và có một “đáp án chuẩn” để đối chiếu khi tối ưu.


In [12]:
def baseline_169(nums):
    freq={}
    for x in nums:
        freq[x]=freq.get(x,0)+1
    for value,times in freq.items():
        if times>len(nums)//2:
            return value
    raise ValueError('Không có majority (ngoài ràng buộc đề)')


In [13]:
for a in [[3,2,3],[2,2,1,1,1,2,2],[-5,-5,-5,3,3],[7]]:
    print(a,'=>',baseline_169(a))


[3, 2, 3] => 3
[2, 2, 1, 1, 1, 2, 2] => 2
[-5, -5, -5, 3, 3] => -5
[7] => 7


### 4.1. Đọc baseline và nhận ra chi phí

Time O(n) trung bình vì duyệt n phần tử và số key không quá n. Extra Space O(k) với k số khác nhau, trường hợp xấu nhất k=O(n). Follow-up yêu cầu Extra Space O(1), nên chỉ dùng Counter chưa giải quyết triệt để. Đổi sang sort rồi lấy phần tử giữa cũng đúng khi majority tồn tại nhưng sort thường O(n log n), có thể thay đổi input nếu sort tại chỗ.

Không nên khẳng định nhanh/chậm chỉ dựa trên việc chạy vài ví dụ. Hãy **đếm bước**, chỉ ra `n` đang đo cái gì và xét input xấu nhất. Sau này khi thử tối ưu, kết quả phải vẫn khớp baseline với cùng một input.


In [14]:
for a in [[3,2,3],[2,2,1,1,1,2,2],[1,2,1,2,1]]:
    print(a,'dict',frequencies_169(a),'mode',baseline_169(a))


[3, 2, 3] dict {3: 2, 2: 1} mode 3
[2, 2, 1, 1, 1, 2, 2] dict {2: 4, 1: 3} mode 2
[1, 2, 1, 2, 1] dict {1: 3, 2: 2} mode 1


### 4.2. Đường vòng / phản ví dụ giúp thấy lỗi tư duy

Nhầm “phần tử phổ biến nhất” với “majority”: `[1,1,2,2,3]` có những phần tử xuất hiện 2/5 và không ai quá bán. Dữ liệu này **ngoài ràng buộc bài gốc**, nhưng rất hữu ích để hiểu vì sao lời hứa majority tồn tại quan trọng. Boyer–Moore chỉ trả một **ứng viên**, không tự chứng minh tần suất ứng viên khi bỏ lời hứa.

Hãy tìm một đầu vào **cực nhỏ** làm giả thuyết sai; dạng test này hiệu quả hơn việc chỉ chạy ngẫu nhiên dữ liệu dài.


In [15]:
outside=[1,1,2,2,3]
print('Ngoài ràng buộc:',outside,'số nhiều nhất',Counter(outside).most_common(1))
print('ngưỡng cần >',len(outside)//2,'nên KHÔNG có majority')
assert max(Counter(outside).values())<=len(outside)//2


Ngoài ràng buộc: [1, 1, 2, 2, 3] số nhiều nhất [(1, 2)]
ngưỡng cần > 2 nên KHÔNG có majority


## 5. Vì sao cần bước cải tiến?

Ta có thể ghép bỏ từng cặp hai giá trị khác nhau: thao tác đó không làm mất majority thực sự. Vì vậy thay vì ghi mọi tần suất, chỉ cần giữ **một ứng viên** và **đếm số phiếu chưa bị triệt**. Khi count về 0, tất cả phần đã đọc có thể coi như đã ghép thành các cặp xung đột, nên ta được quyền nhận ứng viên mới.

Ta không muốn “tối ưu bằng phép màu”. Mục tiêu là **chứng minh có thể bỏ một phần công việc** mà vẫn giữ thông tin đủ để quyết định đúng. Sau khi tìm thấy đại lượng/cấu trúc bắt buộc phải duy trì, ta mới viết thuật toán chính.


### Bước đột phá — Khi gặp phần tử khác, triệt tiêu một phiếu; hết phiếu thì đổi ứng viên

Khởi tạo count=0. Với mỗi x: nếu count==0, đặt candidate=x; sau đó nếu x==candidate thì count+=1, ngược lại count-=1. Không nhất thiết count là số lần candidate có mặt; nó là số phiếu còn lại. Do majority thực sự không thể bị triệt hết, candidate cuối cùng đúng khi đề bảo đảm majority tồn tại.

**Tự kiểm tra:** Nếu làm thêm đúng một bước nữa, cái gì chắc chắn vẫn đúng? Nếu thuật toán dừng ngay bây giờ, vì sao kết quả nó trả là hợp lệ? Hai câu hỏi này chính là hạt nhân của bất biến và điều kiện dừng.


## 6. Thuật toán chính — mổ xẻ từng bước rồi ghép lại

Giải pháp `solve_169` dùng duy nhất `candidate` và `count`, không dùng dict, set, sort. Vòng lặp tuyến tính, mỗi phần tử chỉ xử lý đúng một lần. Trường hợp nums có một phần tử thì count bắt đầu 0, nhận nó làm ứng viên và trả chính giá trị đó.

Trước tiên xem một phép biến đổi ngắn trong cell nhỏ, sau đó đọc hàm `solve_*` không có `print`. **Không nhảy thẳng đến class Solution**: phần luyện tay và trace phía dưới sẽ làm rõ từng điều kiện.


### 6.1. Đọc từng biến, không chỉ đọc tên hàm

- `candidate=None`, `count=0`: chưa giữ phiếu nào.
- `for x in nums`: duyệt tuần tự.
- `if count == 0: candidate=x`: khi không còn phiếu, giá trị mới bắt đầu nhóm chưa triệt tiêu.
- `count += (1 if x==candidate else -1)`: cùng thì nhận phiếu, khác thì hủy một phiếu.
- `return candidate`: đúng nhờ **điều kiện luôn tồn tại majority**, không nhờ count nhất thiết >n//2.

Đặc biệt để ý **thứ tự cập nhật**: đổi thứ tự hai dòng tưởng nhỏ có thể thay đổi ý nghĩa của bất biến. Nếu nhầm, hãy viết trạng thái trước và sau bằng một ví dụ ba phần tử / ba bước.


In [16]:
candidate=2;count=3
for x in [1,2,3]:
    count+=1 if x==candidate else -1
    print('x=',x,'count=',count)
print('count khác với tổng lần xuất hiện của 2:',count)


x= 1 count= 2
x= 2 count= 3
x= 3 count= 2
count khác với tổng lần xuất hiện của 2: 2


In [17]:
def solve_169(nums: list[int]) -> int:
    candidate=None
    count=0
    for x in nums:
        if count==0:
            candidate=x
        count+=1 if x==candidate else -1
    return candidate


### 6.2. Tại sao các nhánh `if`/`while` được viết đúng thứ tự?

Thứ tự `if count==0` **trước** tăng/giảm phiếu bảo đảm một giá trị mới không bị trừ ngay khi chưa có ứng viên. Trong mỗi vòng, count luôn không âm (vì khi count=0, candidate nhận x nên count tăng lên 1). Nếu đầu vào rỗng, hàm không có ý nghĩa — đề đã loại trừ n=0.

Trong một vòng lặp hãy đọc bằng câu tiếng Việt: “**Trước bước này ta đã biết ...; ta kiểm tra ...; nếu ... thì ...; sau bước này ...**”. Có thể thêm `print` để kiểm tra suy luận, nhưng hàm nộp bài phải trả **đúng kiểu** kết quả.


In [18]:
for arr in [[3,2,3],[2,2,1,1,1,2,2],[-3,-3,1,-3,1],[4]]:
    print(arr,'=>',solve_169(arr))
    assert solve_169(arr)==baseline_169(arr)


[3, 2, 3] => 3
[2, 2, 1, 1, 1, 2, 2] => 2
[-3, -3, 1, -3, 1] => -3
[4] => 4


### 6.3. Kiểm tra mấu chốt bằng vài bước tay

Sau khi đọc một tiền tố, các phần tử đã duyệt được chia thành (i) một số **cặp khác giá trị** đã hủy và (ii) `count` phần tử chưa bị hủy, tất cả mang giá trị `candidate`. Điều này đúng khi chưa đọc gì; mỗi x cùng candidate tăng nhóm, x khác candidate ghép hủy một phần tử, khi count=0 nhận nhóm mới.

Một bất biến đúng phải có ba điều: **đúng lúc bắt đầu; vẫn đúng sau một lần cập nhật; khi dừng thì suy ra đáp án**. Dù chưa biết từ “invariant”, bạn có thể phát biểu tự nhiên như câu trên.


In [19]:
cand=None;cnt=0
for x in [1,1,2,2,2,1,2]:
    old=(cand,cnt)
    if cnt==0:cand=x
    cnt+=1 if x==cand else -1
    print('đọc',x,':',old,'->',(cand,cnt))


đọc 1 : (None, 0) -> (1, 1)
đọc 1 : (1, 1) -> (1, 2)
đọc 2 : (1, 2) -> (1, 1)
đọc 2 : (1, 1) -> (1, 0)
đọc 2 : (1, 0) -> (2, 1)
đọc 1 : (2, 1) -> (2, 0)
đọc 2 : (2, 0) -> (2, 1)


## 7. TRACE từng bước — nhìn thấy trạng thái biến

Ta sẽ dùng chương trình `trace_*` khác với lời giải chính: `solve_*` phục vụ submit và test; `trace_*` chỉ phục vụ việc **quan sát**. Trace phải cho thấy dữ liệu đầu vào, biến *trước khi cập nhật*, điều kiện/nhánh đang xét, biến *sau khi cập nhật*, và điều kiện dừng.

Ví dụ `[2,2,1,1,1,2,2]`: có lúc count về 0 rồi candidate đổi; đó không phải bug. Trace phải ghi `candidate/count` **trước khi xử lý x**, rồi hành động gán / tăng / giảm và cặp trạng thái sau.


In [20]:
def trace_169(nums):
    cand=None;cnt=0
    print('i | x | trước(cand,count) | hành động | sau(cand,count)')
    for i,x in enumerate(nums):
        before=(cand,cnt)
        action='nhận ứng viên' if cnt==0 else ('thêm phiếu' if x==cand else 'hủy phiếu')
        if cnt==0:cand=x
        cnt+=1 if x==cand else -1
        print(f'{i:1d} | {x:2d} | {before!s:17} | {action:14s} | {(cand,cnt)}')
    print('ứng viên cuối:',cand)
    return cand
assert trace_169([2,2,1,1,1,2,2])==2


i | x | trước(cand,count) | hành động | sau(cand,count)
0 |  2 | (None, 0)         | nhận ứng viên  | (2, 1)
1 |  2 | (2, 1)            | thêm phiếu     | (2, 2)
2 |  1 | (2, 2)            | hủy phiếu      | (2, 1)
3 |  1 | (2, 1)            | hủy phiếu      | (2, 0)
4 |  1 | (2, 0)            | nhận ứng viên  | (1, 1)
5 |  2 | (1, 1)            | hủy phiếu      | (1, 0)
6 |  2 | (1, 0)            | nhận ứng viên  | (2, 1)
ứng viên cuối: 2


### 7.1. Trace dạng bảng — diễn giải cột và thứ tự cập nhật

Cột `trước` cho biết candidate,count của tiền tố đã xử lý; cột `x` là phần tử sắp xét; cột `hành động` phân biệt nhận ứng viên mới với cộng/triệt phiếu; cột `sau` chính là trạng thái đầu vòng tiếp theo. Khi count=0, **tên candidate cũ không còn đảm bảo gì**.

Khi đọc bảng, đừng coi một hàng là một đáp án; một hàng biểu diễn **một phép chuyển trạng thái**. Hãy tự che cột cuối, dự đoán hành động, rồi đối chiếu với print phía dưới.


In [21]:
assert trace_169([3,2,3])==3
assert trace_169([-1,2,-1,2,-1])==-1


i | x | trước(cand,count) | hành động | sau(cand,count)
0 |  3 | (None, 0)         | nhận ứng viên  | (3, 1)
1 |  2 | (3, 1)            | hủy phiếu      | (3, 0)
2 |  3 | (3, 0)            | nhận ứng viên  | (3, 1)
ứng viên cuối: 3
i | x | trước(cand,count) | hành động | sau(cand,count)
0 | -1 | (None, 0)         | nhận ứng viên  | (-1, 1)
1 |  2 | (-1, 1)           | hủy phiếu      | (-1, 0)
2 | -1 | (-1, 0)           | nhận ứng viên  | (-1, 1)
3 |  2 | (-1, 1)           | hủy phiếu      | (-1, 0)
4 | -1 | (-1, 0)           | nhận ứng viên  | (-1, 1)
ứng viên cuối: -1


### 7.2. Một nhánh đặc biệt mà cách học thuộc code dễ bỏ lỡ

Nếu count về 0 giữa chừng, candidate mới hoàn toàn có thể khác candidate cũ; ví dụ `[1,2,2]`: sau 1 và 2, hai phiếu triệt nhau; 2 còn lại thắng. Nếu chỉ ghi lại ứng viên đầu tiên, bài sẽ sai.

Câu hỏi sau khi chạy: **Nếu bỏ điều kiện xử lý nhánh này, input nhỏ nhất nào trả sai?**


In [22]:
assert trace_169([1,2,2])==2
assert solve_169([1,2,2])==2


i | x | trước(cand,count) | hành động | sau(cand,count)
0 |  1 | (None, 0)         | nhận ứng viên  | (1, 1)
1 |  2 | (1, 1)            | hủy phiếu      | (1, 0)
2 |  2 | (1, 0)            | nhận ứng viên  | (2, 1)
ứng viên cuối: 2


### 7.3. Trường hợp biên — không chỉ là dữ liệu “đẹp”

Trường hợp n=1 trả chính phần tử. Với số âm, dấu của giá trị không ảnh hưởng so sánh `==`. Với dãy `[-1,-1,2]`, majority là -1. Không cần sort hoặc lấy giá trị tuyệt đối.

Trường hợp biên không phải ngoại lệ vô nghĩa: nó bộc lộ một giả định ẩn trong điều kiện lặp hoặc cách tính chỉ số. Chỉ thêm xử lý khi ràng buộc thực sự cho phép trường hợp đó.


In [23]:
for a in [[8],[-1,-1,2],[0,0,1],[-9,4,-9,-9]]:
    assert solve_169(a)==baseline_169(a)
    print(a,'=>',solve_169(a))


[8] => 8
[-1, -1, 2] => -1
[0, 0, 1] => 0
[-9, 4, -9, -9] => -9


### 7.4. Đối chiếu baseline và tối ưu trên một bộ input

So với bảng Counter oracle cho những mảng luôn thỏa lời hứa majority. `count` của Boyer–Moore có thể nhỏ hơn số lần ứng viên thực sự có mặt; **không so count với Counter để chứng minh** vì chúng đo hai đại lượng khác nhau.

Nếu hai lời giải cho kết quả khác nhau, chưa kết luận ngay “baseline sai”; hãy in trạng thái tại input phản ví dụ, đối chiếu lại định nghĩa, rồi sửa mệnh đề bất biến.


In [24]:
for a in [[2,2,1,1,1,2,2],[1,2,1,2,1],[5,5,5,5,1,2,3]]:
    candidate=solve_169(a)
    print('array=',a,'candidate=',candidate,'frequency=',a.count(candidate),'n=',len(a))
    assert candidate==baseline_169(a)


array= [2, 2, 1, 1, 1, 2, 2] candidate= 2 frequency= 4 n= 7
array= [1, 2, 1, 2, 1] candidate= 1 frequency= 3 n= 5
array= [5, 5, 5, 5, 1, 2, 3] candidate= 5 frequency= 4 n= 7


## 8. Vì sao thuật toán đúng? — chứng minh ngắn nhưng đủ ý

**Ý tưởng khử cặp:** trong mảng có majority M xuất hiện f>n/2 lần, loại bỏ một cặp phần tử khác nhau làm tổng phần tử giảm 2, trong khi số lượng M giảm tối đa 1; bất đẳng thức vẫn đúng ở phần còn lại. **Bất biến:** sau mỗi bước, phần đã đọc phân rã thành các cặp khác nhau đã hủy cộng với `count` phiếu chưa hủy đều mang candidate. Khi kết thúc, nếu majority M thực sự tồn tại, M không thể biến mất do các cặp bị hủy; phần dư phải là M. Vì thế `candidate=M`. Nếu bỏ giả thiết majority tồn tại, phải thêm một lần đếm xác minh.

**Phân biệt:** `assert` minh họa “đã chạy đúng các ca này”, không chứng minh hết vô hạn input. Chứng minh dựa vào sự thật được giữ qua mọi lần lặp và điều kiện dừng; đó là kỹ năng sẽ dùng lại ở nhiều bài LeetCode khác.


## 9. Code nộp LeetCode (Python 3)

Đây là phiên bản **không in debug**, trả về đúng dữ liệu cần thiết, giữ đúng tên phương thức `majorityElement` và tên tham số của đề. Khi submit, copy **chỉ** `class Solution` (và import nếu có dùng); với bài dùng `ListNode`, LeetCode đã cung cấp định nghĩa kiểu node.

Sau cell này, notebook sẽ gọi chính `Solution().majorityElement(...)` bằng `assert` để kiểm tra cách đóng gói, tránh trường hợp hàm minh họa đúng nhưng chữ ký nộp sai.


In [25]:
class Solution:
    def majorityElement(self, nums: list[int]) -> int:
        candidate=None
        count=0
        for x in nums:
            if count==0:
                candidate=x
            count+=1 if x==candidate else -1
        return candidate


## 10. Kiểm thử: mẫu đề, biên, phản ví dụ, kiểm thử đối chiếu

Kiểm các mảng odd/even, n=1, số âm, đổi candidate nhiều lần, major nằm cuối và đầu, phần tử nhiễu lặp. Đề bảo đảm majority, nên bộ test submit chỉ gọi với input hợp lệ. Phần ngoài giả thiết sẽ được xử lý tách riêng ở mục biến thể.

Với một ca test, viết rõ **input → expected → actual**. Ưu tiên các ca làm lộ bug thay vì lặp lại hai ví dụ thông thường. Test với input gốc theo đúng kiểu LeetCode, không chỉ kiểm tra các biến đã được biến đổi trong demo.


In [26]:
cases=[([3,2,3],3),([2,2,1,1,1,2,2],2),([7],7),([-1,-1,2],-1),([0,1,0,1,0],0),([1,2,2],2),([9,9,3,9,1,9],9)]
for nums,expected in cases:
    assert Solution().majorityElement(nums)==expected,(nums,expected)
print('Ví dụ, biên và đổi ứng viên:',len(cases),'PASS')


Ví dụ, biên và đổi ứng viên: 7 PASS


### 10.1. Test hệ thống trên một miền nhỏ

Sinh tất cả mảng độ dài 1..8 trên bảng giá trị {0,1,2}, **chỉ giữ mảng có majority**; đối chiếu với Counter. Tối đa có 3^8=6561 chuỗi ở độ dài 8, hoàn toàn phù hợp để phát hiện lỗi cập nhật count và đổi candidate.

Với các input nhỏ, có thể duyệt toàn bộ cấu hình thỏa ràng buộc để tìm phản ví dụ. Nếu không thể duyệt tất cả do số lượng quá lớn, ta kiểm tra toàn bộ một miền con và nêu rõ phạm vi đó, **không gọi nó là chứng minh tổng quát**.


In [27]:
from itertools import product
checked=0
for n in range(1,9):
    for tup in product([0,1,2],repeat=n):
        nums=list(tup)
        c=Counter(nums)
        winners=[x for x,f in c.items() if f>n//2]
        if winners:
            assert len(winners)==1
            assert solve_169(nums)==winners[0]
            checked+=1
print('Vét cạn mọi array hợp lệ độ dài 1..8:',checked,'PASS')


Vét cạn mọi array hợp lệ độ dài 1..8: 3294 PASS


### 10.2. Kiểm thử ngẫu nhiên **có seed** và so sánh với oracle

Để chắc chắn input hợp lệ, tạo một majority x được lặp ít nhất n//2+1 lần, các vị trí còn lại lấy số **khác x**, sau đó shuffle có seed. Oracle đếm độc lập; không vô tình đưa mảng không majority vào bài gốc.

`random.Random(seed)` khiến bộ test có thể chạy lại giống hệt. **Oracle** là một cách kiểm tra độc lập, thường chậm hơn nhưng dễ kiểm chứng. Không dùng hai code giống hệt nhau để đối chiếu — nếu cùng mắc lỗi, test sẽ cho cảm giác an toàn giả.


In [28]:
rng=random.Random(169)
for _ in range(700):
    n=rng.randint(1,150)
    winner=rng.randint(-10,10)
    cnt=rng.randint(n//2+1,n)
    nums=[winner]*cnt
    nums += [winner+1+rng.randint(0,10) for _ in range(n-cnt)]
    rng.shuffle(nums)
    assert solve_169(nums)==baseline_169(nums)==winner
print('700 mảng majority ngẫu nhiên: PASS')


700 mảng majority ngẫu nhiên: PASS


### 10.3. Tính chất khác của đáp án và dữ liệu đầu vào

Nếu một giá trị xuất hiện > n/2 lần, xáo trộn thứ tự mảng không làm thay đổi đáp án, mặc dù `candidate/count` trung gian có thể khác. Đồng thời hàm không được sửa input; ta kiểm tra array vẫn bằng bản copy trước khi gọi.

Không chỉ hỏi “trả đúng số chưa?”; đôi khi đề còn yêu cầu **không sửa cấu trúc** hoặc có ràng buộc độ dài/bit. Kiểm thử thuộc tính giúp phát hiện lỗi chỉ xuất hiện ở nhóm input đặc biệt.


In [29]:
rng=random.Random(2026)
nums=[7]*12 + [1,2,3,4,5,6,8]
original=nums.copy()
for _ in range(100):
    rng.shuffle(nums)
    before=nums.copy()
    assert solve_169(nums)==7
    assert nums==before
print('Hoán vị + không sửa input: PASS')


Hoán vị + không sửa input: PASS


## 11. Phân tích Time / Space Complexity

Gọi **n=len(nums)**. **Baseline với dict/Counter:** Time O(n) trung bình, Extra Space O(k) với k giá trị phân biệt, worst O(n). **Boyer–Moore:** đúng **một vòng for** n lần, mỗi lần so sánh/cộng/trừ O(1), Time O(n), Extra Space O(1) vì chỉ `candidate` và `count`. Nếu không có bảo đảm majority, có thể đếm xác minh candidate bằng một vòng nữa: tổng vẫn O(n) time, O(1) space. Không đưa độ dài của output vào tính Space vì output chỉ là một số.

Để tránh nói Big-O máy móc: chỉ rõ biến kích thước input, số vòng chạy, số bộ nhớ phụ thực sự cấp phát. Nếu đầu ra có độ dài bắt buộc, phân biệt **bộ nhớ phụ** với **bộ nhớ dành cho kết quả**; nếu 32 bit là hằng số, đừng vô tình coi đó là biến tăng vô hạn.


## 12. Sai lầm người mới hay mắc — code sai, vì sao sai, cách sửa

Các lỗi: dùng `>= n//2` thay `>`, giả định count là tần suất thật, không đổi candidate khi count=0, hoặc quên điều kiện bài bảo đảm luôn tồn tại majority. Sorting có thể đúng nhưng tốn O(n log n) và làm thay đổi array nếu `.sort()`.

Không lấy việc có kết quả đúng trên ví dụ đầu làm lý do bỏ qua các lỗi sau. Với từng lỗi, chỉ ra một input phản chứng; chạy đoạn minh họa riêng, sau đó đối chiếu với `solve_*` đã được test.


In [30]:
a=[2,2,1,1,1,2,2]
print('count thực số 2=',a.count(2),'| count Boyer–Moore cuối thường khác')
cand=None;cnt=0
for x in a:
    if cnt==0:cand=x
    cnt+=1 if x==cand else -1
print('candidate=',cand,'phiếu chưa hủy=',cnt)
assert a.count(cand)!=cnt


count thực số 2= 4 | count Boyer–Moore cuối thường khác
candidate= 2 phiếu chưa hủy= 1


### 12.1. Lỗi thứ nhất: đặt `candidate` một lần rồi không bao giờ đổi

Giả sử đọc `[1,2,2]`: nếu lúc đầu candidate=1 rồi count xuống 0, phần đã đọc triệt tiêu. Phải chọn lại ứng viên khi count=0; nếu cứ giữ 1 sẽ trả sai dù 2 chiếm đa số.

**Cách sửa:** quay lại định nghĩa và điều kiện dừng, không chỉ thêm một `if` để vá riêng input mẫu.


In [31]:
def bad_fixed_candidate_169(nums):
    candidate=nums[0];count=0
    for x in nums:count+=1 if x==candidate else -1
    return candidate
print('bad:',bad_fixed_candidate_169([1,2,2]),'| good:',solve_169([1,2,2]))
assert bad_fixed_candidate_169([1,2,2])!=solve_169([1,2,2])


bad: 1 | good: 2


### 12.2. Lỗi thứ hai: tưởng thuật toán tự bảo đảm majority khi đề không hứa

Ở dữ liệu `[1,2,3]`, không có majority; Boyer–Moore có thể trả 3, nhưng đó chỉ là ứng viên chưa được xác minh. Bài gốc tránh trường hợp này bằng ràng buộc. Biến thể tổng quát cần một vòng `nums.count(candidate)>len(nums)//2`.

**Cách sửa:** lập bảng một hoặc hai bước đầu để biết một phép tính dùng trạng thái *trước* hay *sau* cập nhật. Khi đã thấy lỗi, ghi lại một regression test để bug không quay lại.


In [32]:
outside=[1,2,3]
print('ngoài ràng buộc: candidate=',solve_169(outside),'có majority không?',any(x>len(outside)//2 for x in Counter(outside).values()))
assert all(f<=1 for f in Counter(outside).values())


ngoài ràng buộc: candidate= 3 có majority không? False


## 13. Nhận diện pattern và tự suy ra kỹ thuật ở bài khác

Khi đề yêu cầu **phần tử quá bán** và đòi O(n) time / O(1) memory, nghĩ đến **Boyer–Moore** và invariant khử từng cặp khác nhau. Nếu chỉ yêu cầu phần tử xuất hiện nhiều nhất mà **không** bảo đảm quá bán, Boyer–Moore một ứng viên không đủ: phải đổi mục tiêu hoặc xác minh lại. `Counter` vẫn là baseline tốt cho mọi bài đếm.

Một kỹ thuật không gắn với duy nhất bài #169: điều cần ghi nhớ là **dấu hiệu trong đề**, **trạng thái tối thiểu cần giữ**, **quy tắc cập nhật**, **khi nào dừng**, và **mức độ phức tạp có thể đạt được**. Thử tự nói thành 4–5 câu thay vì ghi nhớ code.


## 14. Biến thể — thay đổi yêu cầu để kiểm tra hiểu thật

Biến thể 1: không còn bảo đảm majority; ta thêm một bước xác minh. Biến thể 2: tìm tất cả phần tử xuất hiện >n/3 lần; lúc đó nhiều nhất có hai ứng viên, cần hai bộ đếm (Boyer–Moore mở rộng).

Nếu hiểu thuật toán, bạn có thể thay đổi mục tiêu mà vẫn suy ra cách làm. Phần này chỉ giới thiệu biến thể **có ý nghĩa và nằm ngoài yêu cầu gốc**; code submit phía trên vẫn bám đúng đề LeetCode.


In [33]:
def verified_majority_169(nums):
    if not nums:return None
    candidate=solve_169(nums)
    return candidate if nums.count(candidate)>len(nums)//2 else None
print('ngoài bảo đảm:',verified_majority_169([1,1,2,2,3]))
print('trong bảo đảm:',verified_majority_169([2,2,1]))


ngoài bảo đảm: None
trong bảo đảm: 2


### 14.1. Biến thể thứ nhất: tìm majority khi KHÔNG được bảo đảm tồn tại

Chạy cùng phiếu để chọn candidate; sau đó dùng `nums.count(candidate)` để xác minh. Nếu không quá nửa, trả `None`. Thêm một vòng tuyến tính, vẫn O(n) time / O(1) extra.

Hãy xác định đầu ra mới có cần nhớ thêm trạng thái không. Nếu có, giải thích vì sao; nếu không, chỉ ra biến đang đủ dùng.


In [34]:
for arr in [[],[1,2,3],[1,1,2,2,3],[2,2,3],[5]]:
    print(arr,'->',verified_majority_169(arr))
assert verified_majority_169([1,2,3]) is None
assert verified_majority_169([2,2,3])==2


[] -> None
[1, 2, 3] -> None
[1, 1, 2, 2, 3] -> None
[2, 2, 3] -> 2
[5] -> 5


### 14.2. Biến thể thứ hai: các phần tử xuất hiện nhiều hơn n/3 lần

Một mảng có thể có nhiều nhất **2** giá trị xuất hiện >n/3 lần. Ta giữ hai candidates/counters; mỗi x trùng candidate tăng phiếu, khi có chỗ trống nhận ứng viên mới, còn nếu x khác cả hai thì giảm cả hai. Cuối cùng **bắt buộc đếm lại** để lọc các ứng viên không đạt ngưỡng.

Khi thay đổi ràng buộc đầu vào, các chứng minh cho bài gốc có thể **không còn đúng**; cần sửa chứng minh trước khi sửa code.


In [35]:
def over_third_169(nums):
    a=b=None;c1=c2=0
    for x in nums:
        if a==x and c1>0:c1+=1
        elif b==x and c2>0:c2+=1
        elif c1==0:a=x;c1=1
        elif c2==0:b=x;c2=1
        else:c1-=1;c2-=1
    return [x for x in dict.fromkeys([a,b]) if x is not None and nums.count(x)>len(nums)//3]
print('> n/3:',over_third_169([1,2,3,1,2,1,2]))
assert set(over_third_169([1,2,3,1,2,1,2]))=={1,2}


> n/3: [1, 2]


## 15. Hệ thống luyện tập — 6 bài từ dễ đến nâng cao

**Bài 1 — dự đoán output:** Dự đoán `7//2`, `3 > 7//2` và bảng Counter của `[2,1,2,2,1]`. **Đừng chạy ngay:** ghi kết quả dự đoán, rồi đối chiếu với code nền tảng.

**Bài 2 — tính tay / trace:** Trace candidate,count sau từng x của `[1,2,2,3,2]`; tại đâu count=0 và đổi ứng viên? Ghi ít nhất ba cột *trạng thái trước / thao tác / trạng thái sau*.

**Bài 3 — TODO viết lời giải chính:** hoàn thiện hàm dưới đây mà **không nhìn phần đáp án**.

**Bài 4 — biến thể:** thay đổi kiểu output hoặc điều kiện nhưng dùng lại mấu chốt thuật toán.

**Bài 5 — tự tạo kiểm thử:** nghĩ ít nhất 5 ca phân biệt được code đúng/sai, có ca biên và ca phản chứng.

**Bài 6 — mini challenge:** tự tổng quát hóa kỹ thuật, thử phân tích độ phức tạp và đối chiếu với baseline.

> Các hàm `exercise_*` bên dưới cố ý chưa có lời giải. Chúng **không được gọi tự động** khi chạy notebook. Đáp án chỉ xuất hiện ở mục 17.


### Bài 3 — TODO tự viết lời giải #169

Tự viết `exercise_majority_169(nums)` dùng Boyer–Moore, không `Counter`, không `set`, không `sort`, không sửa nums. Giả thiết luôn có majority, nên không cần bước xác minh.

Gợi ý tối thiểu: trước khi viết, hãy ghi bất biến bằng một câu tiếng Việt; hàm bắt buộc trả đúng loại dữ liệu yêu cầu. Sau khi tự viết, bỏ `pass` và test thủ công với ví dụ và ca biên ở mục 10.


In [36]:
def exercise_majority_169(nums):
    # TODO 1: candidate và count
    # TODO 2: khi count=0, nhận x mới
    # TODO 3: cùng candidate tăng, khác giảm
    # TODO 4: return candidate
    pass


### Bài 4 — TODO biến thể

Viết `exercise_verified_169(nums)` trả majority hoặc None nếu không tồn tại. Đây là **biến thể mở rộng**; có thể cho nums rỗng. Đừng vô tình áp dụng việc “luôn có majority” ở bài gốc.

Bạn được thay kiểu đầu ra nhưng không được thay định nghĩa bài gốc. Để hàm có chữ ký riêng và **không gọi tự động**; tạo dữ liệu thử sau khi đã có code.


In [37]:
def exercise_verified_169(nums):
    # TODO: chọn candidate rồi đếm xác minh; hỗ trợ nums=[]
    pass


### Bài 5 — TODO tự tạo test có chủ đích

Tự thiết kế ít nhất 5 case: n=1, majority số âm, count về 0 rồi đổi candidate, majority nằm ở cuối, ngoài bảo đảm (cho biến thể), và mảng chẵn.

Thử đặt câu hỏi: một chương trình sai phổ biến có thể vẫn vượt qua ca nào? Cần thêm ca nào để khiến nó lộ sai? Viết kiểm thử `assert` trong hàm độc lập, sau đó chỉ gọi khi đã hoàn thiện hàm TODO 3, 4.


In [38]:
def exercise_tests_169():
    # TODO: >=5 assert cho exercise_majority_169 / exercise_verified_169
    pass


### Bài 6 — mini challenge chuyển giao kỹ thuật

Tổng quát Boyer–Moore với ngưỡng >n/3, được phép tối đa 2 winners. Yêu cầu thời gian O(n) và extra space O(1) (ngoài output tối đa 2 phần tử). Bắt buộc xác minh cuối.

Đây là bài nâng cao: bạn không cần giải nhanh. Hãy trình bày ý tưởng bằng tay, viết code sạch, tự mô phỏng input nhỏ rồi mới so sánh với đáp án tham khảo cuối notebook.


In [39]:
def exercise_over_third_169(nums):
    # TODO: hai candidate, hai count; xác minh bằng nums.count
    pass


## 16. Gợi ý ba mức — chỉ mở sau khi tự thử

**Bài 1:** M1: `//` làm tròn xuống. M2: 7//2=3. M3: 3>3 là False; Counter `{2:3,1:2}`.

**Bài 2:** M1: khởi đầu count=0. M2: gặp khác thì hủy phiếu. M3: sau 1 và 2, count=0, x=2 mới nhận candidate 2.

**Bài 3:** M1: lưu cand,count. M2: `if count==0:cand=x`. M3: `count += 1 if x==cand else -1`.

**Bài 4:** M1: ứng viên chưa chắc tồn tại quá bán. M2: `nums.count(cand)`. M3: so `>len(nums)//2`.

**Bài 5:** M1: nhóm n=1, n chẵn, số âm. M2: test đổi candidate. M3: dùng Counter oracle.

**Bài 6:** M1: nhiều nhất hai số >n/3. M2: giữ hai phiếu. M3: xác minh cả hai cuối cùng.

**Cách tự đánh giá:** Nếu mức 1 đã đủ để bạn làm được, hãy che mức 2–3. Nếu phải dùng mức 3, thử giải lại từ đầu sau khi đóng notebook. Học để có thể **tự suy ra**, không phải chép một đoạn code đã biết.


## 17. Đáp án tham khảo — đặt cuối để tránh lộ trước khi làm TODO

Phần này dùng **tên hàm `answer_*` khác với `exercise_*`** để notebook vẫn chạy an toàn. Đáp án không cần giống hệt từng ký tự code của bạn, nhưng phải giữ đúng điều kiện và chứng minh cùng độ phức tạp.

Đáp án bài 3 gồm một vòng duy nhất. Đáp án bài 4 thêm vòng xác minh nếu bỏ lời hứa majority. Đáp án bài 6 cần phân biệt chỉ “ứng viên” và “winner thật”.


In [40]:
def answer_majority_169(nums):
    candidate=None;count=0
    for x in nums:
        if count==0:candidate=x
        count+=1 if x==candidate else -1
    return candidate
assert answer_majority_169([2,2,1,1,1,2,2])==2
print('Đáp án bài 3: PASS')


Đáp án bài 3: PASS


### Đáp án bài 4 — suy ra từ biến thể

Thêm xác minh không phá O(n): một vòng n bước để chọn candidate và một lần `nums.count(candidate)` n bước. Với nums rỗng trả None. Chỉ xét độ xuất hiện thực tế của candidate, không dùng số phiếu chưa triệt.

Hãy so sánh với phần bạn tự viết: code nào dễ nhìn ra bất biến hơn? Cả hai đúng vẫn có thể khác nhau về chi phí bộ nhớ hoặc sự rõ ràng.


In [41]:
def answer_verified_169(nums):
    if not nums:return None
    cand=answer_majority_169(nums)
    return cand if nums.count(cand)>len(nums)//2 else None
for a,ans in [([],None),([1,2,3],None),([2,2,1],2),([1,1,2,2],None)]:
    assert answer_verified_169(a)==ans
print('Đáp án bài 4: PASS')


Đáp án bài 4: PASS


### Đáp án bài 6, và cách kiểm tra bài 5

Sau khi hủy bộ ba ứng viên khác nhau, mọi phần tử xuất hiện >n/3 phải còn thuộc tối đa hai ứng viên. Lọc ngưỡng bằng `count` vì cặp ứng viên có thể cùng là giả. Số phần tử trả không quá 2.

Mỗi test có một mục đích. Khi tự thiết kế test, ghi chú ca nào kiểm điều kiện dừng, ca nào kiểm cập nhật, ca nào kiểm dữ liệu hợp lệ khác thường.


In [42]:
def answer_over_third_169(nums):
    cand1=cand2=None;cnt1=cnt2=0
    for x in nums:
        if cnt1>0 and x==cand1:cnt1+=1
        elif cnt2>0 and x==cand2:cnt2+=1
        elif cnt1==0:cand1=x;cnt1=1
        elif cnt2==0:cand2=x;cnt2=1
        else:cnt1-=1;cnt2-=1
    winners=[]
    for x in [cand1,cand2]:
        if x is not None and x not in winners and nums.count(x)>len(nums)//3:
            winners.append(x)
    return winners

rng=random.Random(16969)
for _ in range(200):
    nums=[rng.randrange(5) for i in range(rng.randint(1,40))]
    oracle={x for x,f in Counter(nums).items() if f>len(nums)//3}
    assert set(answer_over_third_169(nums))==oracle
print('Đáp án bài 6: PASS 200 mẫu')


Đáp án bài 6: PASS 200 mẫu


### Đáp án / lời giải thích bằng lời cho cả 6 bài

**Bài 1:** 7//2=3; 3>3=False; Counter `[2,1,2,2,1]` là `{2:3,1:2}`.

**Bài 2:** Sau x=1 cand=1 count=1; x=2 count=0; x=2 nhận cand=2,count=1; x=3 count=0; x=2 nhận cand=2,count=1.

**Bài 3:** Chọn trạng thái tối thiểu như `solve_*`; đọc lại cell đáp án ngay trên và giải thích từng phép gán.

**Bài 4:** Boyer–Moore chọn ứng viên, `nums.count` xác nhận nó có thật sự quá bán không.

**Bài 5:** Có ca single, chẵn/lẻ, số âm, xung đột phiếu, không majority cho biến thể mở rộng.

**Bài 6:** Với >n/3, giữ hai cặp ứng viên-phiếu, cuối cùng xác minh và trả tối đa hai số.

> Nếu chưa thể tự diễn giải được các dòng `return`, hãy quay lại bảng trace và lặp lại bằng một input tự chọn.


## 18. Tổng kết — template suy nghĩ + checklist nộp bài

### Template nhớ nhanh cho #169

1. **Định nghĩa:** Tìm phần tử xuất hiện >n//2 lần; đề bảo đảm luôn có.
2. **Trạng thái:** Một candidate và count là số phiếu chưa triệt, không phải frequency thật.
3. **Quy tắc cập nhật:** Khi count=0 nhận ứng viên x; cùng giá trị +1, khác giá trị -1.
4. **Bất biến:** Tiền tố đã đọc phân rã thành các cặp khác nhau đã hủy và phiếu candidate còn lại.
5. **Điều kiện dừng:** Duyệt hết nums, trả candidate; nếu bỏ lời hứa tồn tại thì xác minh.
6. **Độ phức tạp:** O(n) time, O(1) extra space.

### Checklist tự đánh giá
- [ ] Tôi phát biểu được đề, output và ràng buộc mà không nhìn code.
- [ ] Tôi hiểu cú pháp mới và tự chạy được ít nhất hai thí nghiệm nhỏ.
- [ ] Tôi trình bày được baseline, điểm yếu và lý do cải tiến.
- [ ] Tôi tự trace được 4 bước, kể cả một nhánh đặc biệt.
- [ ] Tôi phân biệt được chứng minh thuật toán với kiểm thử.
- [ ] Tôi biết khai báo `class Solution` đúng tên hàm, đúng kiểu trả về.
- [ ] Tôi đã làm 6 bài tự luyện **trước khi** xem đáp án.
- [ ] Tôi kiểm tra lại edge cases, time/space, không để print debug khi submit.

**Câu hỏi cuối:** Nếu thay input và biến thể, tôi có tự viết lại thuật toán mà không nhìn notebook được không? Nếu có, bạn đã hiểu kỹ thuật, không chỉ thuộc code.
